In [1]:
import pandas as pd 
import re


In [36]:
test = pd.read_fwf('/Users/nalex2023/main/LSB_main/Review_Paper/papers.txt',
                   delimiter='\t',header=None)


test['DOI_mask'] = test[0].str.contains('https://doi.org/')


test['doi'] = test[0].str.extract(r'(https://doi.org/\S+)')


test



,0,DOI_mask,doi
0,"Abbs DJ, Physick WL (1992) Sea-breeze observat...",False,NaN
1,"Allagbe TD, Guedje FK, Arnaud HVV, Quenum GMLD...",True,https://doi.org/10.4236/acs.2025.152017
2,"Allouche M, Iipponen J, Bou-Zeid E (2025) Unst...",True,https://doi.org/10.1029/2023JD040708
3,"Antonelli M, Rotunno R (2007) Large-Eddy Simul...",True,https://doi.org/10.1175/2007JAS2261.1
4,"Anurose TJ, Subrahamanyam DB, Dutt CBS, KiranK...",True,https://doi.org/10.1007/s00703-011-0178-0
...,...,...,...
154,"Davis, S. R., Farrar, J. T., Weller, R. A., Ji...",True,https://doi.org/10.1029/2019JD031007
155,"Sha, W., Kawamura, T., & Ueda, H. (1991). A Nu...",True,https://doi.org/10.1175/1520-0469(1991)048%253...
156,"Lee, H.-J., Shin, H. H., Lim, K.-S. S., & Park...",True,https://doi.org/10.1016/j.atmosres.2024.107753
157,"Niebler, S., Miltenberger, A., Schmidt, B., & ...",True,https://doi.org/10.5194/wcd-3-113-2022


In [14]:
from pyalex import Works

# Optional: Add your email to get faster response times (OpenAlex "polite pool")
from pyalex import config
config.email = "nmanirmal@gmail.com"

test_doi = test['doi'].dropna().tolist()[0]


test_doi


'https://doi.org/10.4236/acs.2025.152017'

In [15]:
try:
    # Fetch the work by DOI
    work = Works()[f"https://doi.org/{test_doi}"]

    print(f"Title: {work.get('title')}")
    
    # Authors are a list of dictionaries, we extract the display name
    authors = [a['author']['display_name'] for a in work.get('authorships', [])]
    print(f"Authors: {', '.join(authors)}")
    
    # OpenAlex has a specific 'keywords' list
    keywords = [k['display_name'] for k in work.get('keywords', [])]
    print(f"Keywords: {', '.join(keywords)}")

except Exception as e:
    print(f"Error: {e}")

Title: An Automatic Detection Algorithm for Sea Breeze Fronts: A Case Study over the Gulf of Guinea in West Africa
Authors: Thomas D’Aquin Allagbe, François Kossi Guédjé, Blaise Arnaud Hako Touko, Gandomè Mayeul Léger Davy Quenum
Keywords: New guinea, Oceanography, Algorithm, Geology, Geography, Meteorology, Climatology, History, Computer science, Ethnology


In [40]:
test[test['doi'].notna()]['doi']

txt_vals = (test[test['doi'].notna()]['doi'].to_string(index=False))

with open('/Users/nalex2023/main/LSB_main/Review_Paper/doi_list.txt', 'w') as f:
    f.write(txt_vals)
    

In [27]:
final_frame = pd.DataFrame()

final_frame['DOI'] = test['doi'].dropna().tolist()

config.max_retries = 5
config.retry_backoff_factor = 0.5
def fetch_details(doi):
    try:
        work = Works()[f"https://doi.org/{doi}"]
        
        title = work.get('title', 'N/A')
        
        authors = [a['author']['display_name'] for a in work.get('authorships', [])]
        authors_str = ', '.join(authors) if authors else 'N/A'
        
        keywords = [k['display_name'] for k in work.get('keywords', [])]
        keywords_str = ', '.join(keywords) if keywords else 'N/A'
        
        return pd.Series([title, authors_str, keywords_str])
    
    except Exception as e:
        print(f"Error fetching DOI {doi}: {e}")
        return pd.Series(['Error', 'Error', 'Error'])
    

final_frame[['Title', 'Authors', 'Keywords']] = final_frame['DOI'].apply(fetch_details)


Error fetching DOI https://doi.org/10.1175/1520-0450(1993)032%3C0116:EOTLSF%3E2.0.CO;2: 404 Client Error: Not Found for url: https://api.openalex.org/works/https%3A%2F%2Fdoi.org%2Fhttps%3A%2F%2Fdoi.org%2F10.1175%2F1520-0450%281993%29032%253C0116%3AEOTLSF%253E2.0.CO%3B2
Error fetching DOI https://doi.org/10.1175/1520-0493(1995)123%3C0944:OOTSBF%3E2.0.CO;2: 404 Client Error: Not Found for url: https://api.openalex.org/works/https%3A%2F%2Fdoi.org%2Fhttps%3A%2F%2Fdoi.org%2F10.1175%2F1520-0493%281995%29123%253C0944%3AOOTSBF%253E2.0.CO%3B2
Error fetching DOI https://doi.org/10.1175/1520-0493(1995)123%3C3614:SBSADO%3E2.0.CO;2: 404 Client Error: Not Found for url: https://api.openalex.org/works/https%3A%2F%2Fdoi.org%2Fhttps%3A%2F%2Fdoi.org%2F10.1175%2F1520-0493%281995%29123%253C3614%3ASBSADO%253E2.0.CO%3B2
Error fetching DOI https://doi.org/10.1175/1520-0477(1978)059%3C1420:APOTSB%3E2.0.CO;2: 404 Client Error: Not Found for url: https://api.openalex.org/works/https%3A%2F%2Fdoi.org%2Fhttps%3A%2

In [33]:
len(final_frame
    )

133

In [34]:
# print the whole DOI column with spaces between the entries
final_frame_avail = final_frame[final_frame['Title'] != 'Error']


txt_data = (final_frame['DOI'].to_string(index=False))

with open('/Users/nalex2023/main/LSB_main/Review_Paper/final_doi_list.txt', 'w') as f:
    f.write(txt_data)
    


In [35]:
len(final_frame_avail)


86